# Forex Data Analysis

This notebook reads EUR/USD exchange rate data and displays it in a table format.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Read the EUR/USD data from CSV file
df = pd.read_csv('data/old_eurusd.csv')

def percentage(value, total):
    return f"{value / total * 100:.1f}%"

pd.DataFrame({
    'Data': ['Total trades', 'Buy trades', 'EMA', 'BOS'],
    'Value': [df.shape[0], percentage(df[df['Direction'] == 'Buy'].shape[0], df.shape[0]), percentage(df[df['EMA'] == df['Direction']].shape[0], df.shape[0]), percentage(df[df['BOS/CH'] == 'BOS'].shape[0], df.shape[0])]
})

,Data,Value
0,Total trades,933
1,Buy trades,51.0%
2,EMA,64.3%
3,BOS,69.8%


In [2]:
# Display the data table for debugging reasons
df

,Date,Trade,Direction,EMA,SL,Pullback,TP,BOS/CH
0,2025-02-03,#1,Sell,Sell,10.7,10.7,0,BOS
1,2025-02-03,#2,Buy,Buy,5.4,5.4,0,BOS
2,2025-02-03,#4,Buy,Buy,19.3,1.7,24,BOS
3,2025-02-04,#1,Buy,Buy,6.7,0.5,70,BOS
4,2025-02-04,#2,Buy,Sell,7.6,7.6,0,BOS
...,...,...,...,...,...,...,...,...
928,2025-08-08,#2,Sell,Sell,4.0,4.0,0,BOS
929,2025-08-08,#3,Buy,Sell,6.3,6.3,0,CH
930,2025-08-08,#4,Sell,Buy,1.8,1.8,0,CH
931,2025-08-08,#5,Sell,Buy,5.1,5.1,0,CH


In [3]:
# Define RRR calculation function
def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate Risk-Reward Ratio statistics for a given dataframe and strategy.
    
    Args:
        data_df: DataFrame containing trading data
        strategy_name: Name of the strategy for labeling
    
    Returns:
        DataFrame with RRR statistics
    """
    total_trades = len(data_df)
    
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR needs 50% win rate to break even
        (2, 33.0),   # 1:2 RRR needs 33% win rate to break even
        (3, 25.0),   # 1:3 RRR needs 25% win rate to break even
        (4, 20.0),   # 1:4 RRR needs 20% win rate to break even
        (5, 17.0),   # 1:5 RRR needs 17% win rate to break even
        (10, 9.0),   # 1:10 RRR needs 9% win rate to break even
    ]
    
    if total_trades == 0:
        summary_data = {
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
        }
        for ratio, breakeven_rate in rrr_configs:
            summary_data[f'1:{ratio} RRR'] = [0, 0, 0, '0.0%', '0.0%', '0R']
        return pd.DataFrame(summary_data)
    
    
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        edge = win_rate - breakeven_rate
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)

# Define strategy configurations
class Strategy:
    def __init__(self, name, filter_func, description=""):
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """Apply the strategy filter to the dataframe."""
        return self.filter_func(df)

In [4]:
# Define all strategies in simple format
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "All trades without any filtering"
    ),
    # Strategy(
    #     "1pip Pullback Strategy",
    #     lambda df: df[df['Pullback'] >= 1],
    #     "Trades with at least 1 pip pullback"
    # ),
]

In [5]:
# Generic strategy creator with custom lambdas
def create_custom_strategies(configs):
    """
    Generate multiple strategies with custom filter functions.
    
    Args:
        configs: List of tuples containing (name, filter_lambda, description)
                 where filter_lambda is a function that takes (df, value) as parameters
    
    Example:
        configs = [
            (1, lambda df, v: df[df['Pullback'] >= v], "at least {value} pip pullback"),
            (2, lambda df, v: df[df['Pullback'] >= v], "at least {value} pip pullback"),
        ]
    """
    custom_strategies = []
    for value, filter_func, description_template in configs:
        custom_strategies.append(
            Strategy(
                description_template,
                lambda df, v=value, f=filter_func: f(df, v),
                description_template.format(value=value) if '{value}' in description_template else description_template
            )
        )
    return custom_strategies

# Build tons of strategies
strategies.extend(create_custom_strategies([
    # Basic EMA and Direction Strategies
    (1, lambda df, v: df[df['EMA'] == df['Direction']], "EMA + Trade Direction"),
    (2, lambda df, v: df[df['EMA'] != df['Direction']], "EMA + Opposite Trade Direction"),
    (3, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], "EMA + BOS"),
    (4, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')], "EMA + CH"),
    (5, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH')], "Opposite EMA + CH"),
    (6, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'BOS')], "Opposite EMA + BOS"),
    (7, lambda df, v: df[df['BOS/CH'] == 'BOS'], "BOS"),
    (8, lambda df, v: df[df['BOS/CH'] == 'CH'], "CH"),

    # Risk Management Strategies (SL-based entry filters)
    (9, lambda df, v: df[df['SL'] >= 1], "SL >= 1 pip"),
    (10, lambda df, v: df[df['SL'] >= 2], "SL >= 2 pips"),
    (11, lambda df, v: df[df['SL'] >= 5], "SL >= 5 pips"),
    (12, lambda df, v: df[df['SL'] < 1], "SL < 1 pip"),
    (13, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 3)], "SL 1-3 pips"),
    (14, lambda df, v: df[(df['SL'] >= 3) & (df['SL'] <= 7)], "SL 3-7 pips"),
    (15, lambda df, v: df[(df['SL'] >= 7) & (df['SL'] <= 15)], "SL 7-15 pips"),
    (16, lambda df, v: df[df['SL'] >= 15], "SL >= 15 pips"),
    (17, lambda df, v: df[df['SL'] <= 0.5], "SL <= 0.5 pips"),
    (18, lambda df, v: df[df['SL'] >= 10], "SL >= 10 pips"),

    # Advanced Risk Management Strategies
    (19, lambda df, v: df[df['SL'] <= 2], "Conservative: SL <= 2 pips"),
    (23, lambda df, v: df[df['SL'] >= 3], "Moderate Risk: SL >= 3 pips"),
    (24, lambda df, v: df[df['SL'] >= 5], "Aggressive: SL >= 5 pips"),
    (25, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 3)], "Low Risk Zone: SL 1-3 pips"),
    (26, lambda df, v: df[(df['SL'] >= 3) & (df['SL'] <= 6)], "Medium Risk Zone: SL 3-6 pips"),

    # EMA + Direction + SL Combination Strategies
    (27, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] <= 2)], "EMA Aligned + Tight SL"),
    (28, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] >= 3)], "EMA Aligned + Moderate SL"),
    (29, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] <= 2)], "EMA Opposed + Tight SL"),
    (30, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] >= 3)], "EMA Opposed + Moderate SL"),

    # BOS/CH + SL Strategies (Risk-adjusted market structure)
    (31, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "BOS + Conservative SL"),
    (32, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "BOS + Moderate SL"),
    (33, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "CH + Conservative SL"),
    (34, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 3)], "CH + Moderate SL"),

    # BOS/CH + SL Strategies (Duplicate - keeping for completeness)
    (35, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 2)], "BOS + SL >= 2"),
    (36, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 5)], "BOS + SL >= 5"),
    (37, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 2)], "CH + SL >= 2"),
    (38, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 5)], "CH + SL >= 5"),

    # Triple Combination Strategies (SL-based entry filters)
    (39, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "EMA + BOS + Conservative SL"),
    (40, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "EMA + BOS + Moderate SL"),
    (41, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "EMA + CH + Conservative SL"),
    (42, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "Opposite EMA + CH + Conservative SL"),
    (43, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 2)], "EMA + BOS + SL >= 2"),
    (44, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 5)], "EMA + BOS + SL >= 5"),

    # Direction-specific Strategies
    (45, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Buy')], "Buy trades with Buy EMA"),
    (46, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Sell')], "Buy trades with Sell EMA"),
    (47, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Sell')], "Sell trades with Sell EMA"),
    (48, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Buy')], "Sell trades with Buy EMA"),
    (49, lambda df, v: df[(df['Direction'] == 'Buy') & (df['SL'] <= 3)], "Buy trades with Conservative SL"),
    (50, lambda df, v: df[(df['Direction'] == 'Sell') & (df['SL'] <= 3)], "Sell trades with Conservative SL"),

    # High Probability Setup Strategies (SL-based entry filters)
    (51, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], "High Prob: EMA + BOS + Conservative SL"),
    (52, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "High Prob: EMA + BOS + Tight SL"),
    (53, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "Reversal: Opposite EMA + CH + Tight SL"),

    # Risk Management Strategies
    (54, lambda df, v: df[df['SL'] <= 1], "Conservative: SL <= 1 pip"),
    (55, lambda df, v: df[df['SL'] <= 2], "Conservative: SL <= 2 pips"),
    (56, lambda df, v: df[df['SL'] >= 10], "Aggressive: SL >= 10 pips"),
    (57, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 5)], "Moderate Risk: SL 1-5 pips"),

    # Momentum Strategies (SL-based entry timing)
    (20, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] <= 2)], "Momentum: EMA aligned + tight SL"),
    (21, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] >= 4)], "Strong Momentum: EMA aligned + wider SL"),
    (22, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] <= 1)], "Weak Momentum: EMA opposed + tight SL"),

    # Breakout Strategies
    (23, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "Breakout: BOS + tight SL"),
    (24, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "Breakout: BOS + moderate SL"),
    (25, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['EMA'] == df['Direction']) & (df['SL'] <= 3)], "Strong Breakout: BOS + EMA + tight SL"),

    # Reversal Strategies
    (26, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['EMA'] != df['Direction'])], "Reversal: CH + EMA opposed"),
    (27, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 4)], "Reversal: CH + wider SL"),
    (28, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "Reversal: CH + tight SL"),

    # Scalping Strategies
    (29, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 3) & (df['BOS/CH'] == 'BOS')], "Scalp: BOS + SL 1-3"),
    (30, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 1)], "Scalp: BOS + very tight SL"),
    (31, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] <= 2)], "Scalp: EMA aligned + tight SL"),

    # Swing Strategies
    (32, lambda df, v: df[(df['SL'] >= 4) & (df['BOS/CH'] == 'BOS')], "Swing: BOS + wider SL"),
    (33, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 5)], "Swing: BOS + aggressive SL"),
    (34, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] >= 6)], "Swing: EMA aligned + wide SL"),

    # Mean Reversion Strategies
    (35, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] >= 2)], "Mean Reversion: EMA opposed + moderate SL"),
    (36, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 3)], "Mean Reversion: CH + moderate SL"),
    (37, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 3)], "Mean Reversion: Opposed EMA + CH + tight SL"),

    # Trend Following Strategies
    (38, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], "Trend Follow: EMA + BOS"),
    (39, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] >= 4)], "Trend Follow: EMA + wider SL"),
    (40, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['SL'] <= 3)], "Trend Follow: EMA + tight SL"),

    # Contrarian Strategies
    (41, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'BOS')], "Contrarian: Opposed EMA + BOS"),
    (42, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] >= 5)], "Contrarian: Opposed EMA + wide SL"),
    (43, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['SL'] >= 6)], "Contrarian: Opposed EMA + very wide SL"),

    # Volatility-based Strategies
    (44, lambda df, v: df[df['SL'] >= 10], "High Volatility: Wide SL"),
    (45, lambda df, v: df[df['SL'] <= 1], "Low Volatility: Tight SL"),
    (46, lambda df, v: df[(df['SL'] >= 5) & (df['SL'] <= 8)], "Moderate Volatility: SL 5-8"),

    # Complex Combination Strategies
    (47, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], "Complex: EMA+BOS+Tight SL"),
    (48, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2)], "Complex: EMA+CH+Moderate SL"),
    (49, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "Complex: Opposed EMA+BOS+Moderate SL"),
    (50, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 3)], "Complex: Opposed EMA+CH+Tight SL"),

    # Time-based Strategies (London session patterns)
    (51, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "London Momentum: EMA+BOS+Tight SL"),
    (52, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2)], "London Reversal: EMA+CH+Moderate SL"),

    # Risk-Adjusted Strategies (SL-based risk assessment)
    (53, lambda df, v: df[df['SL'] >= 2], "Good Risk Management: SL >= 2"),
    (54, lambda df, v: df[df['SL'] >= 3], "Excellent Risk Management: SL >= 3"),
    (55, lambda df, v: df[df['SL'] < 1], "Poor Risk Management: SL < 1"),
    (56, lambda df, v: df[(df['SL'] >= 1.5) & (df['SL'] <= 4)], "Balanced Risk Management: SL 1.5-4"),

    # Advanced Pattern Strategies
    (57, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['EMA'] == df['Direction']) & (df['SL'] >= 2)], "Pattern: BOS + EMA + Moderate SL"),
    (58, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['EMA'] != df['Direction']) & (df['SL'] >= 1.5)], "Pattern: CH + Opposed EMA + Moderate SL"),
    (59, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "Pattern: BOS + Conservative SL"),
    (60, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 4)], "Pattern: CH + Aggressive SL"),

    # Market Condition Strategies
    (99, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "Trending Market: EMA+BOS+Tight SL"),
    (100, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 3)], "Choppy Market: Opposed EMA+CH+Moderate SL"),

    # Direction-Specific Momentum Strategies
    (61, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Buy') & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], "Bullish Momentum: Buy+EMA+BOS+Tight SL"),
    (62, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Sell') & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], "Bearish Momentum: Sell+EMA+BOS+Tight SL"),
    (63, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Sell') & (df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "Bullish Reversal: Buy+OppEMA+CH+Tight SL"),
    (64, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Buy') & (df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "Bearish Reversal: Sell+OppEMA+CH+Tight SL"),

    # Range Trading Strategies
    (65, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 2) & (df['SL'] <= 4)], "Range Trade: CH + Moderate SL"),
    (66, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] <= 3)], "Range Trade: Opposed EMA + CH + Tight SL"),

    # Break and Retest Strategies
    (67, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 1) & (df['SL'] <= 3)], "Break and Retest: BOS + Moderate SL"),
    (68, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 1) & (df['SL'] <= 4)], "Retest Pattern: CH + Conservative SL"),

    # Volume-Based Strategies (SL as volume proxy)
    (69, lambda df, v: df[(df['SL'] >= 4) & (df['BOS/CH'] == 'BOS')], "High Volume Setup: Wide SL + BOS"),
    (70, lambda df, v: df[(df['SL'] <= 2) & (df['BOS/CH'] == 'CH')], "Low Volume Setup: Tight SL + CH"),

    # Psychological Level Strategies
    (71, lambda df, v: df[(df['SL'] >= 5) & (df['BOS/CH'] == 'BOS')], "Psychological Levels: Wide SL + BOS"),
    (72, lambda df, v: df[(df['SL'] >= 10) & (df['BOS/CH'] == 'BOS')], "Major Psychological Levels: Very Wide SL + BOS"),

    # Market Maker Strategies
    (73, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 2) & (df['SL'] <= 4)], "Market Maker Trap: CH + Moderate SL"),
    (74, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 1.5) & (df['SL'] >= 3)], "Market Maker Stop Hunt: BOS + SL 1.5-3"),

    # Order Flow Strategies
    (75, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 1.5)], "Order Flow: EMA+BOS+Moderate SL"),
    (76, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2)], "Order Flow: Opposed EMA+CH+Moderate SL"),

    # Supply and Demand Strategies
    (77, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['EMA'] == df['Direction']) & (df['SL'] >= 3)], "Supply Zone Break: BOS+EMA+Moderate SL"),
    (78, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['EMA'] != df['Direction']) & (df['SL'] >= 2)], "Demand Zone: CH+OppEMA+Moderate SL"),

    # Multi-Timeframe Strategies
    (79, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 4) & (df['SL'] <= 6)], "Multi-TF: EMA+BOS+SL 4-6"),
    (80, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2) & (df['SL'] <= 5)], "Multi-TF: EMA+CH+SL 2-5"),

    # Seasonal/London Session Strategies
    (81, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Buy') & (df['BOS/CH'] == 'BOS')], "London Buy Momentum"),
    (82, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Sell') & (df['BOS/CH'] == 'BOS')], "London Sell Momentum"),
    (83, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Sell') & (df['BOS/CH'] == 'CH')], "London Buy Reversal"),
    (84, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Buy') & (df['BOS/CH'] == 'CH')], "London Sell Reversal"),

    # Advanced Risk Management Strategies
    (85, lambda df, v: df[df['SL'] <= 0.5], "Ultra Conservative: SL <= 0.5"),
    (86, lambda df, v: df[df['SL'] >= 8], "Very Aggressive: SL >= 8"),
    (87, lambda df, v: df[(df['SL'] >= 0.5) & (df['SL'] <= 1.5)], "Micro Management: SL 0.5-1.5"),
    (88, lambda df, v: df[(df['SL'] >= 6) & (df['SL'] <= 10)], "High Risk Zone: SL 6-10"),

    # Pattern Continuation Strategies
    (89, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 2) & (df['SL'] <= 5)], "Pattern Continuation: EMA+BOS+SL 2-5"),
    (90, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2) & (df['SL'] <= 6)], "Pattern Continuation: EMA+CH+SL 2-6"),

    # Institutional Strategies
    (91, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 6) & (df['EMA'] == df['Direction'])], "Institutional: BOS + Wide SL + EMA Aligned"),
    (92, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 5) & (df['EMA'] != df['Direction'])], "Institutional: CH + Wide SL + EMA Opposed"),

    # Smart Money Strategies
    (93, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "Smart Money: EMA+BOS+Moderate SL"),
    (94, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2.5)], "Smart Money: Opposed EMA+CH+Moderate SL"),

    # Professional Trading Strategies
    (95, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['EMA'] == df['Direction']) & (df['SL'] >= 3) & (df['SL'] <= 5)], "Pro Trade: BOS+EMA+SL 3-5"),
    (96, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['EMA'] != df['Direction']) & (df['SL'] >= 2) & (df['SL'] <= 4)], "Pro Trade: CH+OppEMA+SL 2-4"),

    # Market Structure Strategies
    (97, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 4)], "Market Structure Break: BOS + Wide SL"),
    (98, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 3)], "Market Structure Change: CH + Moderate SL"),

    # Liquidity Strategies
    (99, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 5)], "Liquidity Break: BOS + SL>=5"),
    (100, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 4)], "Liquidity Change: CH + SL>=4"),

    # Final Sophisticated Strategies
    (101, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 2) & (df['SL'] <= 4)], "Ultimate Momentum: EMA+BOS+SL 2-4"),
    (102, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH') & (df['SL'] >= 2) & (df['SL'] <= 5)], "Ultimate Reversal: OppEMA+CH+SL 2-5"),
    (103, lambda df, v: df[(df['Direction'] == 'Buy') & (df['EMA'] == 'Buy') & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 2) & (df['SL'] <= 4)], "Ultimate Bull: Buy+EMA+BOS+SL 2-4"),
    (104, lambda df, v: df[(df['Direction'] == 'Sell') & (df['EMA'] == 'Sell') & (df['BOS/CH'] == 'BOS') & (df['SL'] >= 2) & (df['SL'] <= 4)], "Ultimate Bear: Sell+EMA+BOS+SL 2-4"),
]))

# Calculate all strategy results
strategy_results = {}
for strategy in strategies:
    filtered_df = strategy.apply(df)
    summary_df = calculate_rrr_stats(filtered_df, strategy.name)
    strategy_results[strategy.name] = summary_df

# Function to get top strategies for a specific RRR
def get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5):
    """Get top N strategies for a specific RRR based on outcome."""
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome  # For sorting
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(strategy_performance, key=lambda x: x['outcome_value'], reverse=True)[:top_n]
    
    # Remove the sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# Display top 5 strategies for each RRR in separate tables
display(HTML("<h2>Best Performing Strategies by RRR</h2>"))
rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
    ('1:4 RRR', '1:4'),
    ('1:5 RRR', '1:5'),
    ('1:10 RRR', '1:10')
]
for rrr_column, rrr_label in rrr_configs:
    top_5_df = get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5)
    top_5_df = top_5_df.rename(columns={'Strategy': 'Best ' + rrr_label + ' RRR Strategies'})
    top_5_df = top_5_df.style.set_properties(
        subset=['Best ' + rrr_label + ' RRR Strategies'], 
        **{'width': '220px'}
    )
    display(top_5_df)
    print()  # Add spacing between tables

,Best 1:1 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,High Risk Zone: SL 6-10,275,150,125,54.5%,4.5%,25R
1,SL >= 5 pips,501,262,239,52.3%,2.3%,23R
2,Aggressive: SL >= 5 pips,501,262,239,52.3%,2.3%,23R
3,EMA + BOS + SL >= 5,268,144,124,53.7%,3.7%,20R
4,Swing: EMA aligned + wide SL,250,135,115,54.0%,4.0%,20R


,Best 1:2 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL >= 5 pips,501,188,313,37.5%,4.5%,63R
1,Aggressive: SL >= 5 pips,501,188,313,37.5%,4.5%,63R
2,Moderate Volatility: SL 5-8,325,126,199,38.8%,5.8%,53R
3,High Risk Zone: SL 6-10,275,108,167,39.3%,6.3%,49R
4,EMA + BOS + SL >= 5,268,103,165,38.4%,5.4%,41R


,Best 1:3 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL >= 5 pips,501,146,355,29.1%,4.1%,83R
1,Aggressive: SL >= 5 pips,501,146,355,29.1%,4.1%,83R
2,Moderate Risk: SL >= 3 pips,803,219,584,27.3%,2.3%,73R
3,Excellent Risk Management: SL >= 3,803,219,584,27.3%,2.3%,73R
4,High Risk Zone: SL 6-10,275,86,189,31.3%,6.3%,69R


,Best 1:4 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,High Risk Zone: SL 6-10,275,60,215,21.8%,1.8%,25R
1,Ultimate Bear: Sell+EMA+BOS+SL 2-4,86,22,64,25.6%,5.6%,24R
2,Bearish Momentum: Sell+EMA+BOS+Tight SL,47,14,33,29.8%,9.8%,23R
3,Moderate Volatility: SL 5-8,325,69,256,21.2%,1.2%,20R
4,SL >= 5 pips,501,104,397,20.8%,0.8%,19R


,Best 1:5 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Ultimate Bear: Sell+EMA+BOS+SL 2-4,86,21,65,24.4%,7.4%,40R
1,Bearish Momentum: Sell+EMA+BOS+Tight SL,47,14,33,29.8%,12.8%,37R
2,Scalp: BOS + SL 1-3,101,22,79,21.8%,4.8%,31R
3,Break and Retest: BOS + Moderate SL,101,22,79,21.8%,4.8%,31R
4,Ultimate Momentum: EMA+BOS+SL 2-4,155,31,124,20.0%,3.0%,31R


,Best 1:10 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,EMA + BOS + Conservative SL,32,4,28,12.5%,3.5%,12R
1,High Prob: EMA + BOS + Tight SL,32,4,28,12.5%,3.5%,12R
2,London Momentum: EMA+BOS+Tight SL,32,4,28,12.5%,3.5%,12R
3,Trending Market: EMA+BOS+Tight SL,32,4,28,12.5%,3.5%,12R
4,EMA Aligned + Tight SL,33,4,29,12.1%,3.1%,11R


In [6]:
# Display all strategy results
display(HTML(f"<h2>All Strategies ({len(strategy_results)})</h2>"))

for strategy_name, summary_df in strategy_results.items():
    # Style the first column to have a fixed width
    styled_df = summary_df.style.set_properties(
        subset=[strategy_name], 
        **{'width': '250px'}
    )
    display(styled_df)
    print('')  # Add spacing between tables

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,933,933,933,933,933,933
1,Wins,407,310,247,182,147,30
2,Losses,526,623,686,751,786,903
3,Win Rate,43.6%,33.2%,26.5%,19.5%,15.8%,3.2%
4,Edge,-6.4%,0.2%,1.5%,-0.5%,-1.2%,-5.8%
5,Outcome,-119R,-3R,55R,-23R,-51R,-603R


,EMA + Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,600,600,600,600,600,600
1,Wins,270,207,165,119,96,21
2,Losses,330,393,435,481,504,579
3,Win Rate,45.0%,34.5%,27.5%,19.8%,16.0%,3.5%
4,Edge,-5.0%,1.5%,2.5%,-0.2%,-1.0%,-5.5%
5,Outcome,-60R,21R,60R,-5R,-24R,-369R


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,333,333,333,333,333,333
1,Wins,137,103,82,63,51,9
2,Losses,196,230,251,270,282,324
3,Win Rate,41.1%,30.9%,24.6%,18.9%,15.3%,2.7%
4,Edge,-8.9%,-2.1%,-0.4%,-1.1%,-1.7%,-6.3%
5,Outcome,-59R,-24R,-5R,-18R,-27R,-234R


,EMA + BOS,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,508,508,508,508,508,508
1,Wins,228,176,141,102,80,19
2,Losses,280,332,367,406,428,489
3,Win Rate,44.9%,34.6%,27.8%,20.1%,15.7%,3.7%
4,Edge,-5.1%,1.6%,2.8%,0.1%,-1.3%,-5.3%
5,Outcome,-52R,20R,56R,2R,-28R,-299R


,EMA + CH,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,92,92,92,92,92,92
1,Wins,42,31,24,17,16,2
2,Losses,50,61,68,75,76,90
3,Win Rate,45.7%,33.7%,26.1%,18.5%,17.4%,2.2%
4,Edge,-4.3%,0.7%,1.1%,-1.5%,0.4%,-6.8%
5,Outcome,-8R,1R,4R,-7R,4R,-70R


...

,Liquidity Change: CH + SL>=4,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,203,203,203,203,203,203
1,Wins,104,70,56,40,33,2
2,Losses,99,133,147,163,170,201
3,Win Rate,51.2%,34.5%,27.6%,19.7%,16.3%,1.0%
4,Edge,1.2%,1.5%,2.6%,-0.3%,-0.7%,-8.0%
5,Outcome,5R,7R,21R,-3R,-5R,-181R


,Ultimate Momentum: EMA+BOS+SL 2-4,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,155,155,155,155,155,155
1,Wins,55,49,42,34,31,11
2,Losses,100,106,113,121,124,144
3,Win Rate,35.5%,31.6%,27.1%,21.9%,20.0%,7.1%
4,Edge,-14.5%,-1.4%,2.1%,1.9%,3.0%,-1.9%
5,Outcome,-45R,-8R,13R,15R,31R,-34R


,Ultimate Reversal: OppEMA+CH+SL 2-5,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,76,76,76,76,76,76
1,Wins,27,20,16,11,9,3
2,Losses,49,56,60,65,67,73
3,Win Rate,35.5%,26.3%,21.1%,14.5%,11.8%,3.9%
4,Edge,-14.5%,-6.7%,-3.9%,-5.5%,-5.2%,-5.1%
5,Outcome,-22R,-16R,-12R,-21R,-22R,-43R


,Ultimate Bull: Buy+EMA+BOS+SL 2-4,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,69,69,69,69,69,69
1,Wins,24,21,16,12,10,3
2,Losses,45,48,53,57,59,66
3,Win Rate,34.8%,30.4%,23.2%,17.4%,14.5%,4.3%
4,Edge,-15.2%,-2.6%,-1.8%,-2.6%,-2.5%,-4.7%
5,Outcome,-21R,-6R,-5R,-9R,-9R,-36R


,Ultimate Bear: Sell+EMA+BOS+SL 2-4,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,86,86,86,86,86,86
1,Wins,31,28,26,22,21,8
2,Losses,55,58,60,64,65,78
3,Win Rate,36.0%,32.6%,30.2%,25.6%,24.4%,9.3%
4,Edge,-14.0%,-0.4%,5.2%,5.6%,7.4%,0.3%
5,Outcome,-24R,-2R,18R,24R,40R,2R
